# 第 9 节：函数近似 — Function Approximation

---

## 📍 本节位置

```
Tabular RL (05-08) → **Function Approximation (09)** → DQN (10) → Policy Gradient (11)
                         ↑
                    你在这里
```

到此为止，我们所有的算法（DP、MC、TD、SARSA、Q-Learning）都依赖**查表**：每个状态（或状态-动作对）独立存储一个值。这种方法在状态空间巨大或连续时彻底失效。**函数近似**是 RL 从玩具世界走向真实应用的关键一步。

---

## 🎯 学习目标

1. 理解查表方法的「维度灾难」及其扩展性瓶颈
2. 掌握函数近似的基本形式：$V(s) \approx \hat{V}(s; \mathbf{w})$ 和 $Q(s,a) \approx \hat{Q}(s,a; \mathbf{w})$
3. 理解线性函数近似：特征工程 + 权重向量
4. 掌握**半梯度（semi-gradient）**更新的原理：为何不对 target 求梯度
5. 能区分三种方法（查表、线性 FA、神经网络 FA）的优缺点
6. 理解「致命三角（Deadly Triad）」及其后果


## 1. 为什么查表无法扩展？— When Tables Fail

### 维度灾难

查表法将每个状态视为独立条目。如果是 $D$ 维离散变量、每维 $K$ 个值，状态数就是 $K^D$。当 $D=10, K=10$ 时，这就是 $10^{10} = 100$ 亿个状态。映射到连续空间，这种指数爆炸更无解。

### 连续状态空间的困境

考虑以下常见场景：

- 机器人关节角度：每个关节是连续值，6-DOF 机械臂的状态空间是 $\mathbb{R}^6$ 无界连续空间
- 自动驾驶：车辆位置 (x, y, heading, velocity, steering angle) → 5 维连续空间
- 围棋：$19\times 19$ 棋盘 → $\sim 10^{170}$ 个状态

查表法面对上述任何一个都无法工作。

### 关键问题

| 问题 | 查表法 | 函数近似 |
|------|--------|----------|
| 状态数 | 有限、可枚举 | 无限或巨大 |
| 泛化 | 无 — 每个状态独立 | 有 — 相似状态共享信息 |
| 存储 | O(状态数) | O(参数数) |
| 学习速度 | 慢 — 需逐状态访问 | 快 — 利用结构泛化 |

**核心思想**：与其记住每个状态的值，不如学习一个带参数的函数 $\hat{V}(s; \mathbf{w})$ 来描述价值函数的形状。这样，**新状态的知识可以从相似状态泛化而来**。


## 2. 函数近似的一般形式

### 定义

我们用带参数 $\mathbf{w} \in \mathbb{R}^d$ 的可微函数来近似真实价值函数：

$$\begin{aligned}
V_{\pi}(s) &\approx \hat{V}(s; \mathbf{w}) &\text{(状态价值函数近似)} \\
Q_{\pi}(s, a) &\approx \hat{Q}(s, a; \mathbf{w}) &\text{(动作价值函数近似)}
\end{aligned}$$

其中 $d$ 是参数数量，通常 $d \ll |\mathcal{S}|$。

### 目标

我们希望找到参数 $\mathbf{w}$，使得近似误差最小化：

$$\overline{\text{VE}}(\mathbf{w}) = \sum_{s \in \mathcal{S}} \mu(s) \left[ V_{\pi}(s) - \hat{V}(s; \mathbf{w}) \right]^2$$

其中 $\mu(s)$ 是状态分布（通常是在策略分布）。

### 函数近似器的类型

| 类型 | 形式 | 优点 | 缺点 |
|------|------|------|------|
| **线性** | $\hat{V}(s; \mathbf{w}) = \mathbf{w}^\top \boldsymbol{\phi}(s)$ | 凸优化、理论完善 | 依赖特征工程 |
| **神经网络** | $\hat{V}(s; \mathbf{w}) = \text{NN}(s; \mathbf{w})$ | 自动化特征学习 | 非凸、不稳定 |
| **决策树** | 分段常数函数 | 可解释 | 泛化差 |
| **核方法** | $\hat{V}(s) = \sum_i k(s, s_i) \alpha_i$ | 非线性 | 扩展到大数据困难 |


## 3. 线性函数近似

### 特征表示

线性 FA 假设价值函数是特征向量的线性组合：

$$\hat{V}(s; \mathbf{w}) = \mathbf{w}^\top \boldsymbol{\phi}(s) = \sum_{i=1}^{d} w_i \phi_i(s)$$

其中 $\boldsymbol{\phi}(s) \in \mathbb{R}^d$ 是**特征向量**，$\mathbf{w} \in \mathbb{R}^d$ 是权重参数。

### 特征工程

特征的设计决定了近似能力：

- **多项式特征**：$\phi(s) = [1, s, s^2, s^3, \ldots]$
- **RBF（径向基函数）**：$\phi_i(s) = \exp\left(-\frac{\|s - c_i\|^2}{2\sigma^2}\right)$
- **Tile Coding（拼贴编码）**：将连续空间划分为重叠的网格，每个 tile 是一个二值特征
- **傅里叶特征**：$\phi_i(s) = \cos(i \pi s)$
- **学习到的特征**：神经网络自动提取

### 为什么线性 FA 重要

线性 FA 是 RL 理论学习的重要桥梁：
1. **凸优化**：价值误差 $\overline{\text{VE}}$ 是 $\mathbf{w}$ 的凸函数，保证全局最优
2. **解析解**：对某些目标存在闭式解
3. **理论成熟**：收敛性、误差界的分析非常完善
4. **可解释性**：权重 $w_i$ 直接反映特征 $\phi_i$ 的重要性

下面我们用 GridWorld 来实现线性 FA。


In [ ]:
import os
import numpy as np; import sys; sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from rl_course.utils.seeding import set_seed; set_seed(42)
from rl_course.envs.grid_world import GridWorld
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch; import torch.nn as nn; import os
os.makedirs('outputs/figures', exist_ok=True)
print("Imports OK, torch version:", torch.__version__)


### 3.1 特征构建：Tile Coding

我们在 $5\times 5$ GridWorld 上实现两种特征。由于状态是离散的（$S = \{0, 1, \dots, 24\}$），直接使用 **One-Hot 编码**作为特征：每个状态对应一个独立的二值特征。这等价于查表法，但足以演示线性 FA 的骨架。

随后我们会扩展到连续特征（RBF）来展示泛化能力。


In [ ]:
# ---- 特征构建 ----
class OneHotFeatures:
    """One-hot 特征：每个状态是一个独立的二值特征

    实际上等价于查表法，用于验证线性 FA 的正确性。
    """
    def __init__(self, n_states: int):
        self.n_states = n_states

    def encode(self, state: int) -> np.ndarray:
        """返回 one-hot 特征向量"""
        feat = np.zeros(self.n_states)
        feat[state] = 1.0
        return feat

    @property
    def dim(self) -> int:
        return self.n_states


class RBFGridFeatures:
    """GridWorld 上的 RBF（径向基函数）特征

    在网格上均匀放置 landmark，每个特征是到 landmark 的高斯相似度。
    这允许线性 FA 从已访问状态泛化到未访问状态。
    """
    def __init__(self, width: int, height: int, n_centers: int = 9, sigma: float = 1.5):
        self.width = width
        self.height = height
        self.sigma = sigma
        # 均匀放置中心点
        rows = np.linspace(0, height - 1, int(np.sqrt(n_centers)))
        cols = np.linspace(0, width - 1, int(np.sqrt(n_centers)))
        self.centers = [(r, c) for r in rows for c in cols]
        self.n_centers = len(self.centers)

    def encode(self, state: int) -> np.ndarray:
        r = state // self.width
        c = state % self.width
        feat = np.zeros(self.n_centers)
        for i, (cr, cc) in enumerate(self.centers):
            dist_sq = (r - cr)**2 + (c - cc)**2
            feat[i] = np.exp(-dist_sq / (2 * self.sigma**2))
        return feat

    @property
    def dim(self) -> int:
        return self.n_centers


# 演示：同一个状态（状态 12 = 网格中心），两种特征不同
gw = GridWorld(width=5, height=5)
onehot = OneHotFeatures(gw.n_states)
rbf = RBFGridFeatures(5, 5, n_centers=9, sigma=1.5)

state_center = 12  # 网格中心
print(f"One-hot 特征维度: {onehot.dim}")
print(f"One-hot 编码 (状态 {state_center}): 非零位置 = {np.where(onehot.encode(state_center) > 0)[0]}")
print(f"RBF 特征维度: {rbf.dim}")
print(f"RBF 编码 (状态 {state_center}): {np.round(rbf.encode(state_center), 3)}")

# 可视化：相邻状态的 RBF 特征相似度
state_near = 13  # 右侧邻居
cos_sim = np.dot(rbf.encode(state_center), rbf.encode(state_near)) / (
    np.linalg.norm(rbf.encode(state_center)) * np.linalg.norm(rbf.encode(state_near))
)
print(f"\n状态 {state_center} 和 {state_near} 的 RBF 特征余弦相似度: {cos_sim:.4f}")
print("→ RBF 特征让相似状态具有相似表示，这是泛化的基础")


## 4. 梯度更新与半梯度 — Semi-Gradient

### 全梯度 vs 半梯度

对于线性 FA，我们希望用 TD 方法来更新权重 $\mathbf{w}$。

**TD Target**（bootstrap 目标）：
$$\text{target} = R_{t+1} + \gamma \hat{V}(S_{t+1}; \mathbf{w})$$

**梯度下降更新**（最小化平方误差 $[\text{target} - \hat{V}(S_t; \mathbf{w})]^2$）：

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha [\text{target} - \hat{V}(S_t; \mathbf{w})] \nabla_{\mathbf{w}} \hat{V}(S_t; \mathbf{w})$$

### 关键：为什么是"半梯度"？

如果我们对 $\text{target}$ 也求梯度，那么：

$$\nabla_{\mathbf{w}} [\text{target} - \hat{V}(S_t; \mathbf{w})]^2 = \nabla_{\mathbf{w}} [R_{t+1} + \gamma \hat{V}(S_{t+1}; \mathbf{w}) - \hat{V}(S_t; \mathbf{w})]^2$$

展开后，$\hat{V}(S_{t+1}; \mathbf{w})$ 也出现在梯度中。

**但是我们不这样做！** 原因有二：

1. **Bootstrapping 引入偏差**：target 中的 $\hat{V}$ 本身是有偏估计，对 gradient 贡献梯度会使更新偏向错误的方向
2. **稳定性**：包含 target 的梯度会放大 bootstrapping 带来的误差，容易发散

### 半梯度更新公式

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \left[ R_{t+1} + \gamma \hat{V}(S_{t+1}; \mathbf{w}) - \hat{V}(S_t; \mathbf{w}) \right] \nabla_{\mathbf{w}} \hat{V}(S_t; \mathbf{w})$$

其中 $\nabla_{\mathbf{w}} \hat{V}(S_t; \mathbf{w})$ 的梯度**只作用于** $\hat{V}(S_t; \mathbf{w})$，而不是整个 TD error。

对于**线性 FA**：$\hat{V}(s; \mathbf{w}) = \mathbf{w}^\top \boldsymbol{\phi}(s)$，梯度就是 $\nabla_{\mathbf{w}} \hat{V}(s; \mathbf{w}) = \boldsymbol{\phi}(s)$。

所以半梯度 TD(0) 更新为：
$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \left[ R + \gamma \mathbf{w}^\top \boldsymbol{\phi}(S') - \mathbf{w}^\top \boldsymbol{\phi}(S) \right] \boldsymbol{\phi}(S)$$


In [ ]:
# ---- 半梯度 TD(0) 预测 ----
def semi_gradient_td(
    env, features, gamma=0.99, alpha=0.01, n_episodes=500,
    policy=None
):
    """半梯度 TD(0) 价值预测

    使用线性函数近似: V(s) ≈ w^T φ(s)

    更新规则: w ← w + α [R + γ V(s') - V(s)] φ(s)

    Args:
        env: GridWorld 环境
        features: 特征编码器
        gamma: 折扣因子
        alpha: 学习率
        n_episodes: 训练 episode 数
        policy: 策略函数 (s -> a)，默认随机策略

    Returns:
        w: 训练后的权重向量
        V_history: V(s_start) 随 episode 的变化
    """
    w = np.zeros(features.dim)
    V_history = []

    if policy is None:
        # 默认：均匀随机策略
        def policy(s):
            return env.action_space.sample()

    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False
        total_return = 0.0

        while not done:
            action = policy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 特征编码
            phi_s = features.encode(state)
            phi_ns = features.encode(next_state)

            # 当前 V 估计
            v_s = np.dot(w, phi_s)
            v_ns = np.dot(w, phi_ns)

            # TD error（target 不回传梯度）
            td_error = reward + gamma * v_ns - v_s

            # 半梯度更新（梯度 = φ(s)）
            w += alpha * td_error * phi_s

            state = next_state
            total_return += reward

        # 记录 start 状态的价值
        phi_start = features.encode(env.start_pos[0] * env.width + env.start_pos[1])
        V_history.append(np.dot(w, phi_start))

    return w, V_history


# 在 5x5 GridWorld 上测试
gw = GridWorld(width=5, height=5, start_pos=(0, 0), goal_pos=(4, 4),
               step_reward=-1.0, goal_reward=10.0)

# One-hot 特征（等价查表）
oh_features = OneHotFeatures(gw.n_states)
w_oh, V_oh_hist = semi_gradient_td(gw, oh_features, gamma=0.99, alpha=0.05, n_episodes=500)

# RBF 特征
rbf_features = RBFGridFeatures(5, 5, n_centers=9, sigma=1.5)
w_rbf, V_rbf_hist = semi_gradient_td(gw, rbf_features, gamma=0.99, alpha=0.01, n_episodes=500)

# 对比收敛过程
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(V_oh_hist, label='Linear FA (One-Hot = Tabular)', alpha=0.7)
ax.plot(V_rbf_hist, label='Linear FA (RBF Features)', alpha=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('V(start)')
ax.set_title('Semi-Gradient TD(0) with Linear Function Approximation')
ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig('outputs/figures/09_semi_gradient_td.png', dpi=100, bbox_inches='tight')
plt.close()
print("✅ 半梯度 TD 学习曲线已保存")
print(f"One-Hot V(start) 收敛值: {V_oh_hist[-1]:.3f}")
print(f"RBF V(start) 收敛值: {V_rbf_hist[-1]:.3f}")


## 5. 神经网络作为函数近似器

### 从线性到非线性

线性 FA 的近似能力受限于**特征设计**。神经网络自动学习层次化特征表示：

$$\hat{V}(s; \mathbf{w}) = f_L(f_{L-1}(\cdots f_1(s; \mathbf{w}_1) \cdots ; \mathbf{w}_{L-1}); \mathbf{w}_L)$$

每一层 $f_l$ 是一个非线性变换（通常为 $\text{ReLU}(\mathbf{W}_l \mathbf{h}_{l-1} + \mathbf{b}_l)$）。

### MLP 逼近价值函数

使用 `rl_course.networks.mlp.ValueNetwork`：

```
输入层 (state_dim)
    ↓ Linear + ReLU
隐藏层 (64)
    ↓ Linear + ReLU
隐藏层 (64)
    ↓ Linear (无激活)
输出层 (1) —— V(s) 标量
```

**为什么需要非线性？**
- 线性 FA 只能表示特征空间中的线性函数
- 价值函数通常是非线性的（例如：某些状态的组合效应）
- 神经网络自动学习合适的特征表示

### 对比：三种方法的复杂度

| 方法 | 参数数量 | 表达能力 | 泛化能力 | 理论保证 |
|------|----------|----------|----------|----------|
| 查表 | $\|\mathcal{S}\|$ | 完美（对训练状态） | 无 | 收敛保证 |
| 线性 FA | $d$（特征数） | 线性函数 | 好 | 收敛保证 |
| 神经网络 | $\sum (d_{l-1} \times d_l)$ | 任意连续函数 | 极好 | 需技巧 |


In [ ]:
# ---- 用神经网络做价值近似 ----
from rl_course.networks.mlp import ValueNetwork, QNetwork

# 拿 GridWorld 来演示
gw = GridWorld(width=5, height=5, start_pos=(0, 0), goal_pos=(4, 4),
               step_reward=-1.0, goal_reward=10.0)

# 构建价值网络
# 输入：one-hot 编码（25 维），或直接用状态索引 -> 嵌入
# 这里我们用 one-hot 作为输入
state_dim = gw.n_states
value_net = ValueNetwork(state_dim=state_dim, hidden_dims=[64, 64], activation="relu")

# 测试前向传播
test_states = torch.tensor([[12.0], [13.0], [0.0]])  # 3 个状态
# 转为 one-hot
def to_onehot(states, n_states):
    x = torch.zeros(states.shape[0], n_states)
    x.scatter_(1, states.long(), 1.0)
    return x

x = to_onehot(test_states, state_dim)
with torch.no_grad():
    values = value_net(x).squeeze(-1)

print("测试前向传播：")
for i, s in enumerate([12, 13, 0]):
    print(f"  V(s={s}) = {values[i]:.4f}")

print(f"\n网络参数量: {sum(p.numel() for p in value_net.parameters())}")
print(f"查表法参数量: {gw.n_states}")
print("→ 神经网络用更多参数学到了更灵活的函数形式!")


## 6. 三大方法对比：Tabular vs Linear FA vs Neural Net

我们将三种方法在同样的 GridWorld 任务上比较：
1. **查表法** (TD(0) with table)
2. **线性 FA** (One-Hot features = 等价查表)
3. **线性 FA** (RBF features = 真正的泛化)
4. **神经网络** (MLP ValueNetwork)


In [ ]:
# ---- 完整对比实验 ----
from collections import defaultdict

# ---------- 1. 查表法 TD(0) ----------
def tabular_td(env, gamma, alpha, n_episodes):
    V = np.zeros(env.n_states)
    V_history = []
    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.random.randint(0, env.n_actions)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            td_error = reward + gamma * V[next_state] - V[state]
            V[state] += alpha * td_error
            state = next_state
        s_start = env.start_pos[0] * env.width + env.start_pos[1]
        V_history.append(V[s_start])
    return V, V_history


# ---------- 2. 线性 FA (One-Hot) ----------
def linear_fa_onehot(env, gamma, alpha, n_episodes):
    features = OneHotFeatures(env.n_states)
    w = np.zeros(features.dim)
    V_history = []
    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.random.randint(0, env.n_actions)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            phi_s = features.encode(state)
            phi_ns = features.encode(next_state)
            v_s = np.dot(w, phi_s)
            v_ns = np.dot(w, phi_ns)
            td_error = reward + gamma * v_ns - v_s
            w += alpha * td_error * phi_s
            state = next_state
        s_start = env.start_pos[0] * env.width + env.start_pos[1]
        V_history.append(np.dot(w, features.encode(s_start)))
    return w, V_history


# ---------- 3. 线性 FA (RBF) ----------
def linear_fa_rbf(env, gamma, alpha, n_episodes):
    features = RBFGridFeatures(env.width, env.height, n_centers=9, sigma=1.5)
    w = np.zeros(features.dim)
    V_history = []
    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.random.randint(0, env.n_actions)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            phi_s = features.encode(state)
            phi_ns = features.encode(next_state)
            v_s = np.dot(w, phi_s)
            v_ns = np.dot(w, phi_ns)
            td_error = reward + gamma * v_ns - v_s
            w += alpha * td_error * phi_s
            state = next_state
        s_start = env.start_pos[0] * env.width + env.start_pos[1]
        V_history.append(np.dot(w, features.encode(s_start)))
    return w, V_history


# ---------- 4. 神经网络 FA ----------
def neural_fa(env, gamma, lr, n_episodes):
    state_dim = env.n_states
    net = ValueNetwork(state_dim=state_dim, hidden_dims=[64, 64])
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    V_history = []

    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = np.random.randint(0, env.n_actions)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 转为 one-hot 输入
            def to_tensor(s):
                x = torch.zeros(1, state_dim)
                x[0, s] = 1.0
                return x

            s_t = to_tensor(state)
            ns_t = to_tensor(next_state)

            # 前向传播
            v_s = net(s_t)  # shape (1, 1)
            with torch.no_grad():
                v_ns = net(ns_t)

            # TD target (不梯度)
            target = reward + gamma * v_ns

            # loss = MSE
            loss = nn.MSELoss()(v_s, target)

            # 梯度下降
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            state = next_state

        s_start = env.start_pos[0] * env.width + env.start_pos[1]
        x_start = torch.zeros(1, state_dim)
        x_start[0, s_start] = 1.0
        with torch.no_grad():
            V_history.append(net(x_start).item())

    return net, V_history


# ---------- 运行对比 ----------
gw = GridWorld(width=5, height=5, start_pos=(0, 0), goal_pos=(4, 4),
               step_reward=-1.0, goal_reward=10.0)

print("运行对比实验（共 4 种方法）...")
V_tab, hist_tab = tabular_td(gw, gamma=0.99, alpha=0.05, n_episodes=300)
w_oh, hist_oh = linear_fa_onehot(gw, gamma=0.99, alpha=0.05, n_episodes=300)
w_rbf, hist_rbf = linear_fa_rbf(gw, gamma=0.99, alpha=0.01, n_episodes=300)
net_nn, hist_nn = neural_fa(gw, gamma=0.99, lr=1e-3, n_episodes=300)

print(f"\n最终 V(start) 值:")
print(f"  Tabular TD(0):     {hist_tab[-1]:.4f}")
print(f"  Linear FA (One-Hot): {hist_oh[-1]:.4f}  (应≈查表)")
print(f"  Linear FA (RBF):   {hist_rbf[-1]:.4f}")
print(f"  Neural Net (MLP):  {hist_nn[-1]:.4f}")

# 绘制对比曲线
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hist_tab, label='Tabular TD(0)', linewidth=1.5, alpha=0.8)
ax.plot(hist_oh, label='Linear FA (One-Hot)', linewidth=1.5, alpha=0.8, linestyle='--')
ax.plot(hist_rbf, label='Linear FA (RBF)', linewidth=1.5, alpha=0.8)
ax.plot(hist_nn, label='Neural Net (MLP)', linewidth=1.5, alpha=0.8, linestyle=':')
ax.set_xlabel('Episode'); ax.set_ylabel('V(start)')
ax.set_title('Tabular vs Linear FA vs Neural Net on GridWorld (5x5)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig('outputs/figures/09_fa_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print("\n✅ 三种方法对比图已保存到 outputs/figures/09_fa_comparison.png")


**观察结果分析**：

- Tabular TD 和 One-Hot Linear FA 收敛到相同的值（因为 one-hot + 线性 = 查表）
- RBF Linear FA 由于特征泛化，可能收敛到不同的近似值
- 神经网络 MLP 有更多参数，但需要调整学习率等超参数
- 最简单的问题上，查表法就够用。但状态稍多，就要用 FA


## 7. 价值函数近似质量可视化

让我们直观地检查线性 FA 学到的价值函数的形状，与查表法的解对比。


In [ ]:
# ---- 可视化价值函数 ----
from rl_course.visualization.plotting import plot_value_heatmap

# 1. 查表法结果 (Tabular TD)
V_tab_2d = np.zeros(gw.n_states)
V_tab_2d[:] = V_tab[:]  # 已经训练好了

# 2. 线性 FA (One-Hot) 结果
V_oh_2d = np.zeros(gw.n_states)
for s in range(gw.n_states):
    V_oh_2d[s] = np.dot(w_oh, oh_features.encode(s))

# 3. 线性 FA (RBF) 结果
V_rbf_2d = np.zeros(gw.n_states)
for s in range(gw.n_states):
    V_rbf_2d[s] = np.dot(w_rbf, rbf_features.encode(s))

# 4. 神经网络结果
V_nn_2d = np.zeros(gw.n_states)
state_dim = gw.n_states
for s in range(gw.n_states):
    x = torch.zeros(1, state_dim)
    x[0, s] = 1.0
    with torch.no_grad():
        V_nn_2d[s] = net_nn(x).item()

# 并排对比
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
titles = ['Tabular TD(0)', 'Linear FA (One-Hot)', 'Linear FA (RBF)', 'Neural Net (MLP)']
data_list = [V_tab_2d, V_oh_2d, V_rbf_2d, V_nn_2d]

for ax, V, title in zip(axes, data_list, titles):
    v_grid = V.reshape(gw.height, gw.width)
    im = ax.imshow(v_grid, cmap='YlOrRd', aspect='auto')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    # 标注数值
    for i in range(gw.height):
        for j in range(gw.width):
            ax.text(j, i, f'{v_grid[i, j]:.2f}', ha='center', va='center',
                    fontsize=7, color='black' if v_grid[i, j] > 0 else 'white')

plt.tight_layout()
plt.savefig('outputs/figures/09_value_comparison_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()
print("✅ 价值函数热力图已保存")

# 计算各方法之间的误差
print("\n各方法 vs 查表法 (Tabular) 的 RMS 误差：")
for V, name in zip(data_list[1:], titles[1:]):
    rmse = np.sqrt(np.mean((V - V_tab_2d)**2))
    print(f"  {name}: RMSE = {rmse:.4f}")


## 8. 致命三角 — The Deadly Triad

### 核心洞见

Sutton 在《Reinforcement Learning: An Introduction》中指出，当以下三个要素同时出现时，RL 算法可能**发散**：

```
          ╱╲
         ╱  1  ╲  函数近似 (Function Approximation)
        ╱       ╲
       ╱         ╲
      2 ═══════════ 3
        Bootstrapping
       (TD 更新)
                  ╱╲
                 ╱  4  ╲  Off-Policy 学习
                ╱       ╲
               ╱         ╲
```

| 要素 | 描述 | 例子 |
|------|------|------|
| **函数近似** | 用一个带参数的函数族近似价值函数 | 线性 FA、神经网络 |
| **Bootstrapping** | 用当前估计值更新当前估计值 | TD(0)、n-step TD |
| **Off-Policy 学习** | 从不同于目标策略的分布中学习 | Q-Learning、重要性采样 |

### 为什么致命？

- 任意两个要素可以共存且收敛
- **三个同时出现时**，系统形成**反馈回路**：函数近似的误差通过 bootstrapping 被放大，off-policy 的学习分布又加剧了不稳定性
- 这导致了著名的 **Baird's Counterexample**：一个简单 MDP 上带线性 FA 的 Q-Learning 明确发散

### 应对策略

| 策略 | 方法 | 原理 |
|------|------|------|
| **消除一个要素** | 只用 On-Policy | SARSA 比 Q-Learning 稳定 |
| **增加约束** | Gradient TD | 使用真正的梯度而非半梯度 |
| **经验回放 + 目标网络** | DQN | 打破相关性和稳定 target |
| **控制特征伸缩** | 归一化、正则化 | 防止权重爆炸 |
| **分而治之** | 模型分治 | 分解状态空间 |

> **重要**：致命三角不是理论上的玩具问题。它直接关系到 DQN 等实际算法需要目标网络、经验回放等技巧的原因。


## 9. 批处理 RL 与经验回放预览

### 在线 vs 批处理

到目前为止，我们使用的都是在线的、逐步更新的方法。但这种方式有两个问题：

1. **样本效率低**：每个 transition 只用一次就丢弃
2. **数据相关性强**：连续的 transition 高度相关，不利于梯度更新

### 经验回放 (Experience Replay)

核心思想：用一个**回放缓冲区**存储过去的 transition $(S_t, A_t, R_{t+1}, S_{t+1})$，然后从中随机采样小批量进行更新。

```
采集 → 存入缓冲区 → 随机采样 → 更新网络
  ↑                        │
  └────────────────────────┘
```

### 为什么经验回放有效？

1. **打破相关性**：随机采样消除了连续数据的时间相关性
2. **提高样本效率**：每个 transition 可以被重复使用
3. **平滑更新**：小批量梯度更新比单步更新稳定

### 批处理方法

真正的**批处理 RL**（Batch RL / Fitted Q-Iteration）更进一步：

1. 收集一批数据（固定数据集）
2. 反复在这个数据集上训练
3. 不再与环境交互

这很重要，因为在许多真实场景中（如医疗、自动驾驶）我们无法自由探索。


## 10. 本节总结

### 函数近似的核心思想

$$\boxed{V_{\pi}(s) \approx \hat{V}(s; \mathbf{w}) = \mathbf{w}^\top \boldsymbol{\phi}(s) \quad \text{或} \quad \hat{V}(s; \mathbf{w}) = \text{NN}(s; \mathbf{w})}$$

### 关键公式：半梯度 TD(0) 更新

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \left[ R + \gamma \hat{V}(S'; \mathbf{w}) - \hat{V}(S; \mathbf{w}) \right] \nabla_{\mathbf{w}} \hat{V}(S; \mathbf{w})$$

- **不对 target 求梯度** → 半梯度
- **对于线性 FA**：$\nabla_{\mathbf{w}} \hat{V}(S; \mathbf{w}) = \boldsymbol{\phi}(S)$

### 方法谱系

```
查表法 (Exact, 无法扩展)
    ↓
线性 FA (泛化, 需特征工程)
    ↓
神经网络 (自动化特征学习, 需技巧)
    ↓
深度 RL (DQN, DDPG, PPO, ...)
```

### 取经路线

- 函数近似 = 从"记住答案"到"学习规律"
- 半梯度 = 在理论正确性和实用性之间的优雅平衡
- 致命三角 = 提醒我们 RL 的收敛性未完全解决
- 经验回放和批处理 = 让 RL 从在线走向数据驱动


## 11. 练习

1. **推导**：手推线性 FA 的半梯度 TD(0) 更新公式。从 $\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha \delta_t \nabla \hat{V}(S_t; \mathbf{w}_t)$ 开始，写出 $\delta_t$ 的表达式。

2. **实现**：在 GridWorld 上实现 RBF 特征版的 SARSA（半梯度 SARSA）。对比与 Q-Learning 的稳定性差异。

3. **分析**：为什么轨迹中相邻时间步的数据高度相关？这种相关性对 SGD 有什么影响？写一个小实验来验证。

4. **探索**：尝试不同的 RBF 中心数（如 4、9、16 个中心点）和不同的 sigma，观察对 RBF 线性 FA 近似精度的影响。绘制度量图。

5. **阅读**：查阅 Sutton & Barto 书中关于 Baird's Counterexample 的章节（第 11.2 节）。用 Python 复现这个反例。

6. **思考**：致命三角中的三个要素，如果必须去掉一个来保证稳定性，你会选哪一个？为什么？在 GridWorld 上的实验中，哪个要素最有可能导致不稳定？

7. **扩展**：把 M 维 one-hot 输入改为用 `nn.Embedding` 来表示 GridWorld 状态，然后接 MLP。这种方法与 one-hot + MLP 有何不同？


---
*下一节：[10_dqn.ipynb](10_dqn.ipynb) — Deep Q-Network 与经验回放*
